# Autoencoder for Customer Churn
Predict customer churn using autoencoder-enhanced features on telco data.

## Project Overview
Binary classification predicting telecom customer churn. We use an autoencoder to learn compressed features, then classify with boosting models.

## Learning Objectives
- Apply autoencoders to tabular data
- Predict customer churn from subscription features
- Compare raw vs autoencoder-enhanced features

## Problem Statement
Given a telecom customer's subscription details, predict whether they will churn (leave the service).

## Why This Project Matters
Customer retention is 5-25x cheaper than acquisition. Identifying at-risk customers enables proactive retention campaigns.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Local: WA_Fn-UseC_-Telco-Customer-Churn.csv |
| **Target** | Churn (Yes/No) |
| **Rows** | ~7,043 |
| **Features** | 20 (demographics, services, billing) |

## Dataset Source & License
IBM Sample Data. Telco Customer Churn dataset. Public domain for educational use.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
try: import torch
except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','torch','-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Configuration

In [3]:
LATENT_DIM = 16
EPOCHS = 30
BATCH_SIZE = 64
LR = 1e-3

## Dataset Loading

In [4]:
csv_path = os.path.join(SAVE_DIR, 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(f'Dataset not found at {csv_path}')
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data Validation

In [5]:
print('Missing values:')
print(df.isnull().sum())

TARGET = 'Churn'
print(f'\nTarget: {TARGET}')
print(df[TARGET].value_counts())

# Fix TotalCharges (some blanks)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print(f'\nFixed TotalCharges missing: {df["TotalCharges"].isnull().sum()}')

Missing values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Target: Churn
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Fixed TotalCharges missing: 0


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#2ecc71','#e74c3c'], edgecolor='black')
axes[0,0].set_title('Churn Distribution')

for label in ['Yes', 'No']:
    subset = df[df[TARGET] == label]
    axes[0,1].hist(subset['tenure'], bins=30, alpha=0.5, label=label)
axes[0,1].legend()
axes[0,1].set_title('Tenure by Churn')

for label in ['Yes', 'No']:
    subset = df[df[TARGET] == label]
    axes[1,0].hist(subset['MonthlyCharges'], bins=30, alpha=0.5, label=label)
axes[1,0].legend()
axes[1,0].set_title('Monthly Charges by Churn')

if 'Contract' in df.columns:
    ct = pd.crosstab(df['Contract'], df[TARGET], normalize='index')
    ct.plot.bar(ax=axes[1,1], stacked=True)
    axes[1,1].set_title('Churn by Contract Type')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].value_counts())
print(f'\nChurn rate: {(df[TARGET]=="Yes").mean():.2%}')

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.54%


## Train/Test Split & Preprocessing

In [8]:
# Drop customerID
df = df.drop(columns=['customerID'], errors='ignore')

# Encode target
le_target = LabelEncoder()
df[TARGET] = le_target.fit_transform(df[TARGET])

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c] = LabelEncoder().fit_transform(df[c])

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

# Scale for autoencoder
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (5634, 19), Test: (1409, 19)


## Autoencoder Architecture & Training

In [9]:
input_dim = X_train_scaled.shape[1]

class TabularAE(nn.Module):
    def __init__(self, input_dim, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

    def encode(self, x):
        return self.encoder(x)

model_ae = TabularAE(input_dim).to(device)
optimizer = torch.optim.Adam(model_ae.parameters(), lr=LR)
criterion = nn.MSELoss()

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_scaled)),
                          batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    total_loss = 0
    for (batch,) in train_loader:
        batch = batch.to(device)
        recon = model_ae(batch)
        loss = criterion(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.6f}')

print('Autoencoder training complete.')

Epoch 10/30, Loss: 0.323025


Epoch 20/30, Loss: 0.280951


Epoch 30/30, Loss: 0.262282
Autoencoder training complete.


## Feature Extraction & Combined Features

In [10]:
model_ae.eval()
with torch.no_grad():
    z_train = model_ae.encode(torch.FloatTensor(X_train_scaled).to(device)).cpu().numpy()
    z_test = model_ae.encode(torch.FloatTensor(X_test_scaled).to(device)).cpu().numpy()

# Combine original + latent features
X_train_combined = pd.DataFrame(
    np.hstack([X_train.values, z_train]),
    columns=list(X_train.columns) + [f'z{i}' for i in range(LATENT_DIM)]
)
X_test_combined = pd.DataFrame(
    np.hstack([X_test.values, z_test]),
    columns=list(X_train.columns) + [f'z{i}' for i in range(LATENT_DIM)]
)

# Use combined features for classification
X_train_final = X_train_combined
X_test_final = X_test_combined
print(f'Combined features: {X_train_final.shape[1]} ({X_train.shape[1]} original + {LATENT_DIM} latent)')

Combined features: 35 (19 original + 16 latent)


## Baseline Model

In [11]:
bl = LogisticRegression(max_iter=1000, random_state=SEED)
bl.fit(X_train_final, y_train)
bl_pred = bl.predict(X_test_final)
print(f'Baseline LR (combined features): Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}')

Baseline LR (combined features): Acc=0.7984  F1=0.5860


## LazyPredict Benchmark

In [12]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train_final))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train_final.head(n_lp), X_test_final.head(min(1000, len(X_test_final))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                               Accuracy  Balanced Accuracy   ROC AUC  \
Model                                                                  
NearestCentroid                   0.705           0.723993  0.798719   
QuadraticDiscriminantAnalysis     0.721           0.718365  0.802221   
AdaBoostClassifier                0.787           0.710230  0.818793   
GaussianNB                        0.716           0.706672  0.797608   
BernoulliNB                       0.719           0.701625  0.792824   
LGBMClassifier                    0.791           0.697588  0.820975   
LogisticRegression                0.774           0.694254  0.820526   
CatBoostClassifier                0.781           0.687209  0.818518   
LinearDiscriminantAnalysis        0.768           0.684243  0.812329   
LinearSVC                         0.773           0.682928  0.816530   
XGBClassifier                     0.766           0.676964  0.804046   
RidgeClassifier                   0.774           0.676515  0.81

## FLAML AutoML

In [13]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train_final, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test_final)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Accuracy={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [14]:
models = {}
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train_final, y_train)
models['CatBoost'] = cb

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train_final, y_train)
models['LightGBM'] = lgb

xgb = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train_final, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test_final)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p):.4f}")

CatBoost: Acc=0.7864  F1=0.5554
LightGBM: Acc=0.7786  F1=0.5517
XGBoost: Acc=0.7864  F1=0.5593


## Top 3 Model Selection

In [15]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test_final)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p),
        'Precision': precision_score(y_test, p),
        'Recall': recall_score(y_test, p),
    }
results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

          Accuracy        F1  Precision    Recall
XGBoost   0.786373  0.559297   0.618123  0.510695
CatBoost  0.786373  0.555391   0.620462  0.502674
LightGBM  0.778566  0.551724   0.596273  0.513369

Top 3: ['XGBoost', 'CatBoost', 'LightGBM']


## Final Evaluation of Top 3

In [16]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test_final)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p),
        'Precision': precision_score(y_test, p),
        'Recall': recall_score(y_test, p),
    }
    print(f"{name}:")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

XGBoost:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.62      0.51      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

CatBoost:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.62      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

LightGBM:
              precision    recall  f1-score   support

           0       0.83      0.87      0.85      1035
           1       0.60      0.51      0.55       374

    accuracy                           0.78      1409
   macro avg       0.71      0.69      0.70      1409
weighted avg       0.77      0.78      0.77  

## Error Analysis

In [17]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test_final)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train_final.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

err_rate = (preds != y_test.values).sum() / len(y_test)
print(f'Error rate: {err_rate:.4f}')

Error rate: 0.2136


## Interpretation & Business Insight
Tenure, contract type, and monthly charges are top churn predictors. Month-to-month contracts have highest churn risk. Autoencoder features add marginal improvement.

## Limitations
- Small dataset (7K rows)
- Autoencoder may not add much on structured tabular data
- No temporal churn dynamics

## How to Improve
- Variational autoencoder for better latent space
- Survival analysis for time-to-churn
- Add customer interaction data

## Production Considerations
- Score customers monthly for churn risk
- Trigger retention offers for high-risk customers
- Monitor model drift as products change

## Common Mistakes
- Not fixing TotalCharges type conversion
- Overfitting autoencoder on small data
- Ignoring that trees already handle feature interactions

## Mini Challenge
1. Compare accuracy with and without autoencoder features
2. Try LATENT_DIM = 8 vs 32
3. Visualize autoencoder reconstructions

## Final Summary
Combined autoencoder features with traditional classifiers for churn prediction. Boosting models perform well; autoencoder adds compressed representations.

In [18]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "XGBoost": {
    "Accuracy": 0.7864,
    "F1": 0.5593,
    "Precision": 0.6181,
    "Recall": 0.5107
  },
  "CatBoost": {
    "Accuracy": 0.7864,
    "F1": 0.5554,
    "Precision": 0.6205,
    "Recall": 0.5027
  },
  "LightGBM": {
    "Accuracy": 0.7786,
    "F1": 0.5517,
    "Precision": 0.5963,
    "Recall": 0.5134
  },
  "best_model": "XGBoost"
}
